#환경

In [ ]:
!pip -q install "qai-hub-models[facemap-3dmm]" mediapipe opencv-python-headless scikit-learn joblib pillow
!wget -q -O /content/face_landmarker.task \
  https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.9/97.9 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.6/503.6 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 195.4/195.4 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 133.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

##학습용 영상 전처리

In [ ]:
from google.colab import files
import os

os.makedirs("/content/train_videos", exist_ok=True)
uploaded = files.upload()
for name in uploaded.keys():
    os.rename(name, f"/content/train_videos/{name}")

print(os.listdir("/content/train_videos"))


[]


In [ ]:
import os
import glob
import subprocess
from pathlib import Path

VIDEO_DIR = "/content/train_videos"
FRAME_ROOT = "/content/train_frames"
FPS = 2

os.makedirs(FRAME_ROOT, exist_ok=True)

video_paths = sorted(glob.glob(f"{VIDEO_DIR}/*"))
print("videos:", video_paths)

for video_path in video_paths:
    stem = Path(video_path).stem
    out_dir = f"{FRAME_ROOT}/{stem}"
    os.makedirs(out_dir, exist_ok=True)
    cmd = [
        "ffmpeg", "-y",
        "-i", video_path,
        "-vf", f"fps={FPS}",
        "-q:v", "2",
        f"{out_dir}/frame_%05d.jpg",
    ]
    subprocess.run(cmd, check=True)

print("done")


videos: []
done


#전처리

##import + helper

In [ ]:
import os
import glob
import json
import joblib
import cv2
import numpy as np
import torch
import mediapipe as mp

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error

from qai_hub_models.models.facemap_3dmm.model import FaceMap_3DMM

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

def create_face_landmarker(model_path="/content/face_landmarker.task", num_faces=1):
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=num_faces,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    return FaceLandmarker.create_from_options(options)

def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def collect_image_paths(root):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    return sorted(paths)

def infer_mediapipe_teacher(landmarker, image_rgb):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = landmarker.detect(mp_image)
    if not result.face_landmarks or not result.face_blendshapes:
        return None

    h, w = image_rgb.shape[:2]
    pts = np.array(
        [[lm.x * w, lm.y * h, lm.z * w] for lm in result.face_landmarks[0]],
        dtype=np.float32,
    )
    score_map = {c.category_name: float(c.score) for c in result.face_blendshapes[0]}

    matrix = None
    if result.facial_transformation_matrixes:
        matrix = np.array(result.facial_transformation_matrixes[0], dtype=np.float32).reshape(-1)

    return pts, score_map, matrix

def get_face_bbox_from_landmarks(pts, image_shape, padding=0.25, make_square=True):
    h, w = image_shape[:2]
    xs = pts[:, 0]
    ys = pts[:, 1]

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    bw = x1 - x0
    bh = y1 - y0

    x0 -= bw * padding
    x1 += bw * padding
    y0 -= bh * padding
    y1 += bh * padding

    if make_square:
        cx = (x0 + x1) / 2.0
        cy = (y0 + y1) / 2.0
        size = max(x1 - x0, y1 - y0)
        x0 = cx - size / 2.0
        x1 = cx + size / 2.0
        y0 = cy - size / 2.0
        y1 = cy + size / 2.0

    x0 = int(np.clip(np.floor(x0), 0, w - 1))
    y0 = int(np.clip(np.floor(y0), 0, h - 1))
    x1 = int(np.clip(np.ceil(x1), 0, w - 1))
    y1 = int(np.clip(np.ceil(y1), 0, h - 1))
    return x0, y0, x1, y1

def infer_qualcomm_265(model, image_rgb, bbox):
    x0, y0, x1, y1 = bbox
    crop = image_rgb[y0:y1 + 1, x0:x1 + 1]
    if crop.size == 0:
        return None

    crop = cv2.resize(crop, (128, 128), interpolation=cv2.INTER_LINEAR)
    inp = torch.from_numpy(crop).float() / 255.0
    inp = inp.permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        raw = model(inp)[0].cpu().numpy().astype(np.float32)

    return raw  # 보통 265차원


##데이터셋


In [ ]:
FRAME_ROOT = "/content/train_frames"
image_paths = collect_image_paths(FRAME_ROOT)
print("num frames:", len(image_paths))

q_model = FaceMap_3DMM.from_pretrained()

X_list = []
Y_list = []
used_paths = []
skipped = []
blendshape_names = None

MAX_ABS_YAW = 0.6   # 정면/준정면 위주
MIN_FACE_W = 50
MIN_FACE_H = 50

with create_face_landmarker("/content/face_landmarker.task") as landmarker:
    for i, path in enumerate(image_paths):
        try:
            image_rgb = read_rgb(path)

            teacher = infer_mediapipe_teacher(landmarker, image_rgb)
            if teacher is None:
                skipped.append((path, "mediapipe_failed"))
                continue

            pts, score_map, matrix = teacher

            bbox = get_face_bbox_from_landmarks(
                pts,
                image_rgb.shape,
                padding=0.25,
                make_square=True,
            )

            bw = bbox[2] - bbox[0]
            bh = bbox[3] - bbox[1]
            if bw < MIN_FACE_W or bh < MIN_FACE_H:
                skipped.append((path, "face_too_small"))
                continue

            x = infer_qualcomm_265(q_model, image_rgb, bbox)
            if x is None:
                skipped.append((path, "qualcomm_failed"))
                continue

            # Qualcomm yaw는 보통 259번, rad 변환
            yaw_rad = float(x[259]) * float(np.pi / 2.0)
            if abs(yaw_rad) > MAX_ABS_YAW:
                skipped.append((path, "yaw_filtered"))
                continue

            if blendshape_names is None:
                blendshape_names = sorted(score_map.keys())

            y = np.array([score_map[name] for name in blendshape_names], dtype=np.float32)

            X_list.append(x)
            Y_list.append(y)
            used_paths.append(path)

            if (i + 1) % 200 == 0:
                print(f"processed {i+1}/{len(image_paths)}")

        except Exception as e:
            skipped.append((path, str(e)))

X = np.stack(X_list).astype(np.float32)
Y = np.stack(Y_list).astype(np.float32)

np.savez(
    "/content/facemap_coeff_to_mp52_dataset.npz",
    X=X,
    Y=Y,
    blendshape_names=np.array(blendshape_names, dtype=object),
    used_paths=np.array(used_paths, dtype=object),
)

with open("/content/facemap_coeff_to_mp52_skipped.json", "w") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print("X:", X.shape)
print("Y:", Y.shape)
print("saved: /content/facemap_coeff_to_mp52_dataset.npz")


num frames: 0


ValueError: need at least one array to stack

#학습


In [ ]:
data = np.load("/content/facemap_coeff_to_mp52_dataset.npz", allow_pickle=True)
X = data["X"]
Y = data["Y"]
blendshape_names = data["blendshape_names"].tolist()

X_train, X_val, y_train, y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)

model = Ridge(alpha=3.0)
model.fit(X_train_s, y_train)

pred_val = np.clip(model.predict(X_val_s), 0.0, 1.0)

overall_mae = mean_absolute_error(y_val, pred_val)
print("overall MAE:", overall_mae)

for i, name in enumerate(blendshape_names):
    mae_i = np.mean(np.abs(pred_val[:, i] - y_val[:, i]))
    print(f"{name:24s} mae={mae_i:.4f}")

joblib.dump(model, "/content/facemap_coeff_to_mp52_ridge.joblib")
joblib.dump(scaler, "/content/facemap_coeff_to_mp52_scaler.joblib")

with open("/content/facemap_coeff_to_mp52_names.json", "w") as f:
    json.dump(blendshape_names, f, ensure_ascii=False, indent=2)

print("saved model files")


overall MAE: 0.022278418764472008
_neutral                 mae=0.0000
browDownLeft             mae=0.0412
browDownRight            mae=0.0429
browInnerUp              mae=0.0530
browOuterUpLeft          mae=0.0397
browOuterUpRight         mae=0.0312
cheekPuff                mae=0.0001
cheekSquintLeft          mae=0.0000
cheekSquintRight         mae=0.0000
eyeBlinkLeft             mae=0.0354
eyeBlinkRight            mae=0.0356
eyeLookDownLeft          mae=0.0443
eyeLookDownRight         mae=0.0429
eyeLookInLeft            mae=0.0225
eyeLookInRight           mae=0.0327
eyeLookOutLeft           mae=0.0313
eyeLookOutRight          mae=0.0210
eyeLookUpLeft            mae=0.0332
eyeLookUpRight           mae=0.0311
eyeSquintLeft            mae=0.0360
eyeSquintRight           mae=0.0421
eyeWideLeft              mae=0.0064
eyeWideRight             mae=0.0096
jawForward               mae=0.0003
jawLeft                  mae=0.0011
jawOpen                  mae=0.0167
jawRight                 mae=0

#내 meta에

In [ ]:
import json
import numpy as np
import joblib

METADATA_PATH = "/content/metadata_1775803704587.json"
OUTPUT_PATH = "/content/avatar_player_multi_face.json"

model = joblib.load("/content/facemap_coeff_to_mp52_ridge.joblib")
scaler = joblib.load("/content/facemap_coeff_to_mp52_scaler.joblib")

with open("/content/facemap_coeff_to_mp52_names.json", "r") as f:
    blendshape_names = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

frames = metadata["frames"]

pts_list = [int(fr["pts_us"]) for fr in frames if "pts_us" in fr]
if len(pts_list) >= 2:
    deltas = np.diff(np.array(pts_list, dtype=np.int64))
    deltas = deltas[deltas > 0]
    fps = float(1_000_000.0 / np.median(deltas)) if len(deltas) else 30.0
else:
    fps = 30.0

start_pts = int(frames[0]["pts_us"]) if frames else 0
input_dim = scaler.mean_.shape[0]

def bbox_to_xyxy(bbox):
    if not bbox:
        return None
    x = float(bbox["x"])
    y = float(bbox["y"])
    w = float(bbox["width"])
    h = float(bbox["height"])
    return [x, y, x + w, y + h]

MAX_ABS_YAW = 0.6
MIN_FACE_W = 50
MIN_FACE_H = 50

results = []

for i, frame in enumerate(frames):
    out_faces = []

    for face in frame.get("faces", []):
        bbox = face.get("bbox", {})
        bw = float(bbox.get("width", 0))
        bh = float(bbox.get("height", 0))
        if bw < MIN_FACE_W or bh < MIN_FACE_H:
            continue

        coeff = np.asarray(face["tdmm_raw"]["coeffs"], dtype=np.float32).reshape(-1)
        if coeff.shape[0] < input_dim:
            continue

        yaw_rad = float(coeff[259]) * float(np.pi / 2.0)
        if abs(yaw_rad) > MAX_ABS_YAW:
            continue

        x = coeff[:input_dim]
        x_s = scaler.transform(x[None])
        pred = np.clip(model.predict(x_s)[0], 0.0, 1.0)

        blendshape_map = {
            name: float(score)
            for name, score in zip(blendshape_names, pred)
        }

        out_faces.append({
            "tracking_id": int(face.get("tracking_id", -1)),
            "bbox": bbox_to_xyxy(bbox),
            "pose_radians": {
                "pitch": float(coeff[258]) * float(np.pi / 2.0),
                "yaw": float(coeff[259]) * float(np.pi / 2.0),
                "roll": float(coeff[260]) * float(np.pi / 2.0),
            },
            "blendshapes": blendshape_map,
        })

    results.append({
        "frame_index": i,
        "time_sec": float((int(frame["pts_us"]) - start_pts) / 1_000_000.0),
        "faces": out_faces,
    })

payload = {
    "fps": fps,
    "frames": results,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("saved:", OUTPUT_PATH)


saved: /content/avatar_player_multi_face.json


In [ ]:
import json
import numpy as np
import joblib

METADATA_PATH = "/content/metadata_1775803704587.json"
OUTPUT_PATH = "/content/avatar_player_multi_face.json"

model = joblib.load("/content/facemap_coeff_to_mp52_ridge.joblib")
scaler = joblib.load("/content/facemap_coeff_to_mp52_scaler.joblib")

with open("/content/facemap_coeff_to_mp52_names.json", "r") as f:
    blendshape_names = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

frames = metadata["frames"]

pts_list = [int(fr["pts_us"]) for fr in frames if "pts_us" in fr]
if len(pts_list) >= 2:
    deltas = np.diff(np.array(pts_list, dtype=np.int64))
    deltas = deltas[deltas > 0]
    fps = float(1_000_000.0 / np.median(deltas)) if len(deltas) else 30.0
else:
    fps = 30.0

start_pts = int(frames[0]["pts_us"]) if frames else 0
input_dim = scaler.mean_.shape[0]

MAX_ABS_YAW = 0.6
MIN_FACE_W = 50
MIN_FACE_H = 50

BASE_THRESHOLD = 0.02
EYE_THRESHOLD = 0.05
EYE_SMOOTH_ALPHA = 0.75
EYE_MAX_STEP = 0.12

EYE_KEYS = [
    "eyeBlinkLeft", "eyeBlinkRight",
    "eyeSquintLeft", "eyeSquintRight",
    "eyeWideLeft", "eyeWideRight",
    "eyeLookDownLeft", "eyeLookDownRight",
    "eyeLookInLeft", "eyeLookInRight",
    "eyeLookOutLeft", "eyeLookOutRight",
    "eyeLookUpLeft", "eyeLookUpRight",
]

prev_eye_by_track = {}

def bbox_to_xyxy(bbox):
    if not bbox:
        return None
    x = float(bbox["x"])
    y = float(bbox["y"])
    w = float(bbox["width"])
    h = float(bbox["height"])
    return [x, y, x + w, y + h]

results = []

for i, frame in enumerate(frames):
    out_faces = []

    for face in frame.get("faces", []):
        bbox = face.get("bbox", {})
        bw = float(bbox.get("width", 0))
        bh = float(bbox.get("height", 0))
        if bw < MIN_FACE_W or bh < MIN_FACE_H:
            continue

        coeff = np.asarray(face["tdmm_raw"]["coeffs"], dtype=np.float32).reshape(-1)
        if coeff.shape[0] < input_dim:
            continue

        yaw_rad = float(coeff[259]) * float(np.pi / 2.0)
        if abs(yaw_rad) > MAX_ABS_YAW:
            continue

        x = coeff[:input_dim]
        x_s = scaler.transform(x[None])
        pred = np.clip(model.predict(x_s)[0], 0.0, 1.0)

        blendshape_map = {}
        for name, score in zip(blendshape_names, pred):
            val = float(score)
            threshold = EYE_THRESHOLD if name in EYE_KEYS else BASE_THRESHOLD
            if val < threshold:
                val = 0.0
            blendshape_map[name] = val

        tracking_id = int(face.get("tracking_id", -1))
        prev_eye = prev_eye_by_track.get(tracking_id)

        if prev_eye is not None:
            for name in EYE_KEYS:
                curr = float(blendshape_map.get(name, 0.0))
                prev_val = float(prev_eye.get(name, curr))

                smoothed = EYE_SMOOTH_ALPHA * prev_val + (1.0 - EYE_SMOOTH_ALPHA) * curr

                if smoothed > prev_val + EYE_MAX_STEP:
                    smoothed = prev_val + EYE_MAX_STEP
                elif smoothed < prev_val - EYE_MAX_STEP:
                    smoothed = prev_val - EYE_MAX_STEP

                if smoothed < EYE_THRESHOLD:
                    smoothed = 0.0

                blendshape_map[name] = float(np.clip(smoothed, 0.0, 1.0))

        prev_eye_by_track[tracking_id] = {
            name: float(blendshape_map.get(name, 0.0))
            for name in EYE_KEYS
        }

        out_faces.append({
            "tracking_id": tracking_id,
            "bbox": bbox_to_xyxy(bbox),
            "pose_radians": {
                "pitch": float(coeff[258]) * float(np.pi / 2.0),
                "yaw": float(coeff[259]) * float(np.pi / 2.0),
                "roll": float(coeff[260]) * float(np.pi / 2.0),
            },
            "blendshapes": blendshape_map,
        })

    results.append({
        "frame_index": i,
        "time_sec": float((int(frame["pts_us"]) - start_pts) / 1_000_000.0),
        "faces": out_faces,
    })

payload = {
    "fps": fps,
    "frames": results,
}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

print("saved:", OUTPUT_PATH)


saved: /content/avatar_player_multi_face.json


#새로운 영상

##영상 업로드 및 프레임


In [ ]:
from google.colab import files
import os

uploaded = files.upload()
video_name = next(iter(uploaded.keys()))

os.makedirs("/content/infer_video", exist_ok=True)
os.rename(video_name, f"/content/infer_video/{video_name}")

VIDEO_PATH = f"/content/infer_video/{video_name}"
print("video:", VIDEO_PATH)


Saving d0.mp4 to d0.mp4
video: /content/infer_video/d0.mp4


In [ ]:
!mkdir -p /content/infer_frames
!ffmpeg -y -i "$VIDEO_PATH" -vf fps=3 -q:v 2 /content/infer_frames/frame_%05d.jpg
!ls /content/infer_frames | head
!ls /content/infer_frames | wc -l

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

##모델로드

In [ ]:
import os
import glob
import json
import joblib
import cv2
import numpy as np
import torch
import mediapipe as mp

from PIL import Image
from qai_hub_models.models.facemap_3dmm.model import FaceMap_3DMM

BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

def create_face_landmarker(model_path="/content/face_landmarker.task", num_faces=5):
    options = FaceLandmarkerOptions(
        base_options=BaseOptions(model_asset_path=model_path),
        running_mode=VisionRunningMode.IMAGE,
        num_faces=num_faces,
        min_face_detection_confidence=0.5,
        min_face_presence_confidence=0.5,
        min_tracking_confidence=0.5,
        output_face_blendshapes=True,
        output_facial_transformation_matrixes=True,
    )
    return FaceLandmarker.create_from_options(options)

def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))

def collect_image_paths(root):
    exts = ["*.jpg", "*.jpeg", "*.png", "*.webp", "*.bmp"]
    paths = []
    for ext in exts:
        paths.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    return sorted(paths)

def infer_mediapipe_faces(landmarker, image_rgb):
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image_rgb)
    result = landmarker.detect(mp_image)

    if not result.face_landmarks:
        return []

    h, w = image_rgb.shape[:2]
    faces = []

    for idx, landmarks in enumerate(result.face_landmarks):
        pts = np.array(
            [[lm.x * w, lm.y * h, lm.z * w] for lm in landmarks],
            dtype=np.float32,
        )

        matrix = None
        if result.facial_transformation_matrixes and idx < len(result.facial_transformation_matrixes):
            matrix = np.array(result.facial_transformation_matrixes[idx], dtype=np.float32).reshape(-1)

        faces.append({
            "pts": pts,
            "matrix": matrix,
        })

    return faces

def get_face_bbox_from_landmarks(pts, image_shape, padding=0.25, make_square=True):
    h, w = image_shape[:2]
    xs = pts[:, 0]
    ys = pts[:, 1]

    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()

    bw = x1 - x0
    bh = y1 - y0

    x0 -= bw * padding
    x1 += bw * padding
    y0 -= bh * padding
    y1 += bh * padding

    if make_square:
        cx = (x0 + x1) / 2.0
        cy = (y0 + y1) / 2.0
        size = max(x1 - x0, y1 - y0)
        x0 = cx - size / 2.0
        x1 = cx + size / 2.0
        y0 = cy - size / 2.0
        y1 = cy + size / 2.0

    x0 = int(np.clip(np.floor(x0), 0, w - 1))
    y0 = int(np.clip(np.floor(y0), 0, h - 1))
    x1 = int(np.clip(np.ceil(x1), 0, w - 1))
    y1 = int(np.clip(np.ceil(y1), 0, h - 1))

    return x0, y0, x1, y1

def infer_qualcomm_265(model, image_rgb, bbox):
    x0, y0, x1, y1 = bbox
    crop = image_rgb[y0:y1 + 1, x0:x1 + 1]

    if crop.size == 0:
        return None

    crop = cv2.resize(crop, (128, 128), interpolation=cv2.INTER_LINEAR)
    inp = torch.from_numpy(crop).float() / 255.0
    inp = inp.permute(2, 0, 1).unsqueeze(0)

    with torch.no_grad():
        raw = model(inp)[0].cpu().numpy().astype(np.float32)

    return raw

q_model = FaceMap_3DMM.from_pretrained()
reg = joblib.load("/content/facemap_coeff_to_mp52_ridge.joblib")
scaler = joblib.load("/content/facemap_coeff_to_mp52_scaler.joblib")

with open("/content/facemap_coeff_to_mp52_names.json", "r") as f:
    blendshape_names = json.load(f)

print("ready")


ready


##변환

In [ ]:
frame_paths = collect_image_paths("/content/infer_frames")
print("num frames:", len(frame_paths))

FPS_VALUE = 3.0
MIN_FACE_W = 50
MIN_FACE_H = 50
MAX_ABS_YAW = 0.6
THRESHOLD = 0.02

results = []
skipped = []

with create_face_landmarker("/content/face_landmarker.task", num_faces=5) as landmarker:
    for i, path in enumerate(frame_paths):
        try:
            image_rgb = read_rgb(path)
            detected_faces = infer_mediapipe_faces(landmarker, image_rgb)

            frame_faces = []

            for face in detected_faces:
                pts = face["pts"]

                bbox = get_face_bbox_from_landmarks(
                    pts,
                    image_rgb.shape,
                    padding=0.25,
                    make_square=True,
                )

                bw = bbox[2] - bbox[0]
                bh = bbox[3] - bbox[1]
                if bw < MIN_FACE_W or bh < MIN_FACE_H:
                    continue

                coeff = infer_qualcomm_265(q_model, image_rgb, bbox)
                if coeff is None:
                    continue

                yaw_rad = float(coeff[259]) * float(np.pi / 2.0)
                if abs(yaw_rad) > MAX_ABS_YAW:
                    continue

                x = coeff[:scaler.mean_.shape[0]]
                x_s = scaler.transform(x[None])
                pred = np.clip(reg.predict(x_s)[0], 0.0, 1.0)

                pred_map = {}
                for name, score in zip(blendshape_names, pred):
                    val = float(score)
                    if val < THRESHOLD:
                        val = 0.0
                    pred_map[name] = val

                frame_faces.append({
                    "bbox": [int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3])],
                    "pose_radians": {
                        "pitch": float(coeff[258]) * float(np.pi / 2.0),
                        "yaw": float(coeff[259]) * float(np.pi / 2.0),
                        "roll": float(coeff[260]) * float(np.pi / 2.0),
                    },
                    "blendshapes": pred_map,
                })

            results.append({
                "frame_index": i,
                "time_sec": i / FPS_VALUE,
                "faces": frame_faces,
            })

            if (i + 1) % 100 == 0:
                print(f"done {i+1}/{len(frame_paths)}")

        except Exception as e:
            skipped.append((path, str(e)))
            results.append({
                "frame_index": i,
                "time_sec": i / FPS_VALUE,
                "faces": [],
            })

payload = {
    "fps": FPS_VALUE,
    "frame_size": {
        "width": int(read_rgb(frame_paths[0]).shape[1]),
        "height": int(read_rgb(frame_paths[0]).shape[0]),
    },
    "frames": results,
}

with open("/content/new_video_avatar_player.json", "w") as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

with open("/content/new_video_avatar_player_skipped.json", "w") as f:
    json.dump(skipped, f, ensure_ascii=False, indent=2)

print("saved: /content/new_video_avatar_player.json")
print("saved: /content/new_video_avatar_player_skipped.json")


num frames: 59
saved: /content/new_video_avatar_player.json
saved: /content/new_video_avatar_player_skipped.json


##저장

In [ ]:
from google.colab import files
files.download("/content/new_video_avatar_player.json")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>